In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
from crewai_tools import SerperDevTool

serper_dev_tool = SerperDevTool()


In [15]:
import yaml
from pprint import pprint

# Load YAML Configuration
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

pprint(config)

{'agents': {'fact_checker_agent': {'backstory': 'You specialize in detecting '
                                                'misinformation and validating '
                                                'claims using credible '
                                                'sources.',
                                   'goal': 'Verify research findings and '
                                           'ensure factual accuracy.',
                                   'role': 'Fact-Checking Specialist'},
            'research_agent': {'backstory': 'You are a skilled researcher with '
                                            'expertise in retrieving credible, '
                                            'real-time information from online '
                                            'sources.',
                               'goal': 'Find the most relevant and up-to-date '
                                       'information on a given topic.',
                               'role':

In [16]:
from crewai import Agent, Task

research_agent = Agent(
    role=config["agents"]["research_agent"]["role"],
    goal=config["agents"]["research_agent"]["goal"],
    backstory=config["agents"]["research_agent"]["backstory"],
    tools=[serper_dev_tool],
    verbose=True
)

research_task = Task(
    description=config["tasks"]["research_task"]["description"],
    agent=research_agent,
    tools=[serper_dev_tool],
    expected_output=config["tasks"]["research_task"]["expected_output"]
)

In [17]:
# Summarizer agent
summarization_agent = Agent(
    role=config["agents"]["summarization_agent"]["role"],
    goal=config["agents"]["summarization_agent"]["goal"],
    backstory=config["agents"]["summarization_agent"]["backstory"],
    verbose=True
)

summarization_task = Task(
    description=config["tasks"]["summarization_task"]["description"],
    agent=summarization_agent,
    expected_output=config["tasks"]["summarization_task"]["expected_output"],
)

In [18]:
# Fact checker agent
fact_checker_agent = Agent(
    role=config["agents"]["fact_checker_agent"]["role"],
    goal=config["agents"]["fact_checker_agent"]["goal"],
    backstory=config["agents"]["fact_checker_agent"]["backstory"],
    tools=[serper_dev_tool],
    verbose=True
)

fact_checking_task = Task(
    description=config["tasks"]["fact_checking_task"]["description"],
    agent=fact_checker_agent,
    tools=[serper_dev_tool],
    expected_output=config["tasks"]["fact_checking_task"]["expected_output"],
)

In [ ]:
# Crew
from crewai import Crew, Process

research_crew = Crew(
    agents=[research_agent, summarization_agent, fact_checker_agent],
    tasks=[research_task, summarization_task, fact_checking_task],
    process=Process.sequential, # ensures tasks run in order -> Research, summarize, fact check
    verbose=True
)


# Note: Notebooks already run inside an asyncio event loop, so use the async
# entry point and await it instead of the synchronous kickoff().
result = await research_crew.kickoff_async(inputs={"topic": "The impact of AI on job markets"})
print("\nFinal Verified Summary:\n", result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 35ff51a7-383a-4a6b-a1a0-066e4ee1e840                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the SerperDevTool to find the most relevant and recent data on The impact of AI on job markets.      │
│  ID: 261dae32-b0ab-484f-b5ac-864453a18272                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internet Researcher                                                                                     │
│                                                                                                                 │
│  Task: Use the SerperDevTool to find the most relevant and recent data on The impact of AI on job markets.      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#11) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'The impact of AI on job markets 2024'}                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_the_internet_with_serper executed with result: {'searchParameters': {'q': 'The impact of AI on job markets 2024', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': '[PDF] Artificial Intelligence Impact on Labor Markets', 'lin...

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Checking Specialist                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here is the fact-checked and verified complete research report on "The Impact of AI on Job Markets: A 2024     │
│  Research Report," based on the latest credible sources as verified:                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## The Impact of AI on Job Markets: A 2024 Research Report                                                     │
│                                                                                                                 │
│  ### 1. Overview                                                                                                │
│  Artificial Intelligence (AI) continues to be a transformative force reshaping the labor market globally. Its   │
│  impact spans potential job displacement through automation, alteration in job roles, creation of new sectors   │
│  and positions, productivity enhancement, and shifts in labor demand patterns.                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Key Insights from Leading Reports and Research                                                          │
│                                                                                                                 │
│  #### A. Artificial Intelligence Impact on Labor Markets (IEDC PDF)                                             │
│  - AI automates routine tasks and augments capabilities in complex job roles.                                   │
│  - Key concerns include possible job displacement, but new roles focused on managing and improving AI systems   │
│  are emerging.                                                                                                  │
│  - The labor market is expected to see growing demand for technology-driven roles while repetitive and manual   │
│  jobs face higher automation risk.                                                                              │
│  - Reference: [IEDC AI Impact on Labor Markets                                                                  │
│  PDF](https://www.iedconline.org/clientuploads/EDRP%20Logos/AI_Impact_on_Labor_Markets.pdf)                     │
│                                                                                                                 │
│  #### B. Goldman Sachs Analysis: How Will AI Affect the US Labor Market?                                        │
│  - Goldman Sachs estimates approximately 300 million jobs worldwide could be exposed to AI automation over the  │
│  next 10 years.                                                                                                 │
│  - Generative AI is predicted to increase labor productivity by around 15%.                                     │
│  - While some occupations may experience reduced payrol

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Cross-check the summarized information for accuracy and remove any misleading claims.                    │
│  Agent: Fact-Checking Specialist                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 35ff51a7-383a-4a6b-a1a0-066e4ee1e840                                                                       │
│  Final Output: Here is the fact-checked and verified complete research report on "The Impact of AI on Job       │
│  Markets: A 2024 Research Report," based on the latest credible sources as verified:                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## The Impact of AI on Job Markets: A 2024 Research Report                                                     │
│                                                                                                                 │
│  ### 1. Overview                                                                                                │
│  Artificial Intelligence (AI) continues to be a transformative force reshaping the labor market globally. Its   │
│  impact spans potential job displacement through automation, alteration in job roles, creation of new sectors   │
│  and positions, productivity enhancement, and shifts in labor demand patterns.                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Key Insights from Leading Reports and Research                                                          │
│                                                                                                                 │
│  #### A. Artificial Intelligence Impact on Labor Markets (IEDC PDF)                                             │
│  - AI automates routine tasks and augments capabilities in complex job roles.                                   │
│  - Key concerns include possible job displacement, but new roles focused on managing and improving AI systems   │
│  are emerging.                                                                                                  │
│  - The labor market is expected to see growing demand for technology-driven roles while repetitive and manual   │
│  jobs face higher automation risk.                                                                              │
│  - Reference: [IEDC AI Impact on Labor Markets                                                                  │
│  PDF](https://www.iedconline.org/clientuploads/EDRP%20Logos/AI_Impact_on_Labor_Markets.pdf)                     │
│                                                                                                                 │
│  #### B. Goldman Sachs Analysis: How Will AI Affect the US Labor Market?                                        │
│  - Goldman Sachs estimates approximately 300 million jobs worldwide could be exposed to AI automation over the  │
│  next 10 years.                                                                                                 │
│  - Generative AI is predicted to increase labor productivity by around 15%.                                     │
│  - While some occupations may experience reduced payro


Final Verified Summary:
 Here is the fact-checked and verified complete research report on "The Impact of AI on Job Markets: A 2024 Research Report," based on the latest credible sources as verified:

---

## The Impact of AI on Job Markets: A 2024 Research Report

### 1. Overview  
Artificial Intelligence (AI) continues to be a transformative force reshaping the labor market globally. Its impact spans potential job displacement through automation, alteration in job roles, creation of new sectors and positions, productivity enhancement, and shifts in labor demand patterns.

---

### 2. Key Insights from Leading Reports and Research

#### A. Artificial Intelligence Impact on Labor Markets (IEDC PDF)  
- AI automates routine tasks and augments capabilities in complex job roles.  
- Key concerns include possible job displacement, but new roles focused on managing and improving AI systems are emerging.  
- The labor market is expected to see growing demand for technology-driven roles whil

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯